# SWE-Opt Quick Start

This notebook demonstrates SWE-Opt from prompt construction through inference and evaluation.

SWE-Opt supports two ways to generate patches:

- **Single-pass inference:** build an oracle/BM25 prompt and send it to the model once.
- **Agentic inference:** give OpenCode a task-only prompt and let it inspect, edit, and test the repository iteratively inside Docker.

Choose a path in section 1. Both produce patches that can be evaluated by the same harness.

Use this notebook to explore prompt settings, LLM providers, and evaluation configurations.

> **Note**
> For inference or evaluation runs, copy the printed commands and run them directly in a terminal.

## 0 · Reproducible setup with uv

Install [uv](https://docs.astral.sh/uv/getting-started/installation/), then run from the repository root:

```bash
uv python install
uv sync --locked --extra notebook
uv run --locked --extra notebook jupyter lab quick_start.ipynb
```

Select the project's `.venv` Python kernel in VS Code. Python is pinned by `.python-version`, and exact dependencies are recorded in `uv.lock`. Use `uv run --locked python -m ...` for terminal commands instead of an unrelated system Python.

For direct OpenAI inference, add `--extra openai` to `uv sync`; other provider extras are documented in the README. OpenCode agentic inference does not require a host provider SDK. Do not regenerate the lockfile during routine runs.

The host environment alone is not enough for full experiment reproducibility: also preserve dataset versions, baseline image digests, OpenCode version, model settings, prompts, and seeds. Docker and external model services are not installed or pinned by uv.

## 1 · Choose an execution path

Choose the style of model behavior you want to evaluate. The two paths differ in the context given to the model and who controls the next action; both produce patches for the same evaluation harness.

| Path | How it works | Go to |
|---|---|---|
| **A · Single-pass** | Build an oracle/BM25 prompt and ask the model once for a patch. | Section 2 |
| **B · Agentic** | Give OpenCode a task-only prompt and let it explore, edit, and test iteratively inside Docker. | Section 3 |

## 2 · Path A — Single-pass inference

This path builds an oracle or BM25 prompt, sends it to the selected model in one inference run, and produces patches for evaluation. Follow these steps for Path A; Path B is described after the inference cell.

### 2.1 · Prompt Config

In [ ]:
DATASET_PATH  = "SWE-Opt/SWE-Opt"
OUTPUT_PATH   = "data/prompt_dataset/out.json"
TOKENIZER     = "cl100k_base"
MAX_TOKENS    = 200_000

### 2.2 · Build Oracle Prompts

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "-m",
    "swe_opt.prep.prompt.prompt_oracle_builder",
    "--dataset_path", DATASET_PATH,
    "--output_path", OUTPUT_PATH,
    "--tokenizer", TOKENIZER,
    "--max_tokens", str(MAX_TOKENS)
]

print("Running:", " ".join(cmd))

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

### 2.3 · Build BM25 Prompts

In [ ]:
import subprocess
import sys

#BM25_RETRIEVALS_PATH = ""

cmd = [
    sys.executable,
    "-m",
    "swe_opt.prep.prompt.prompt_bm25_builder",
    "--dataset_path",         DATASET_PATH,
    "--bm25_retrievals_path", BM25_RETRIEVALS_PATH,
    "--output_path",          OUTPUT_PATH,
    "--tokenizer",            TOKENIZER,
    "--max_tokens",           str(MAX_TOKENS),
]

print("Running:", " ".join(cmd))
print("Using python:", sys.executable)

result = subprocess.run(cmd, capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

### 2.4 · Inspect

In [ ]:
from swe_opt.prep.data_loader import DatasetLoader

loader  = DatasetLoader(OUTPUT_PATH, mode="raw")
samples = list(loader)
print(f"Total samples: {len(samples)}")

In [ ]:
# Preview one prompt
row = samples[0]
print("scenario_id:", row.get("scenario_id"))
print()
print(row["text"][:1000])

In [ ]:
from swe_opt.prep.utils import prompt_preview

# Preview a specific scenario_id.
# If `scenario_id` is None or not found, the first sample is used.

id, text = prompt_preview(
    input_json=OUTPUT_PATH,
    selected_scenario_id=None  # or "some_scenario_id"
)

print("id:", id)
print("text:", text)

### 2.5 · Inference

Runs the inference pipeline on the prompt dataset built in the previous step.
Completions are saved to `inference/<run_id>/instances/<scenario_id>/completion.txt`.
Already-completed scenarios are skipped automatically on re-run.

> **Note on API keys**
>
> Before running inference, make sure the required API key is available as an environment variable or in a `.env` file in the project root.
>
> The current API key mapping is:
>
> - `openai_chat` -> `OPENAI_API_KEY`
> - `openai_responses` -> `OPENAI_API_KEY`
> - `hf` -> `HUGGINGFACE_API_KEY`
> - `ollama` -> `OLLAMA_API_KEY`
>
> Example `.env` file:
>
> ```bash
> OPENAI_API_KEY=your_openai_key_here
> HUGGINGFACE_API_KEY=your_hf_key_here
> OLLAMA_API_KEY=your_ollama_key_here
> ```

> **Adding a new LLM clients**
>
> The inference module uses a factory-based LLM client design under `inference/llm_client/`.
> Existing clients include:
>
> - `openai_chat_client.py`
> - `openai_responses_client.py`
> - `ollama_client.py`
> - `hf_client.py`
>
> All clients follow the same structure:
>
> 1. inherit from `BaseLLMClient`
> 2. implement `generate(prompt: str) -> str`
> 3. register the provider in `get_llm_client(...)`
> 4. add API key support in `_load_key_from_env(...)` if needed
>
> If you want to add another LLM client, follow the same recipe:
>
> - create a new client file under `inference/llm_client/`
> - implement a new class based on `BaseLLMClient`
> - add the provider branch in `get_llm_client(...)`
> - add a new environment variable entry in `_load_key_from_env(...)` if the provider needs authentication
>
> This keeps the inference pipeline modular and consistent across providers.

> **Note on running inference**
>
> The notebook prints the full command used to run inference.  
> For better performance and stability, we recommend copying this printed command and running it directly in a **terminal** instead of executing it inside the notebook.
>
> Running the command in a terminal is typically faster and avoids potential notebook-related overhead during long inference runs.

In [ ]:
import subprocess
import sys

# Make sure to add your API key to a `.env` file or set it as an environment variable.

cmd = [
    sys.executable,
    "-m",
    "swe_opt.inference.run_inference",
    "--dataset_path", OUTPUT_PATH,   # from prompt preparation cell
    "--provider", "openai_chat",
    "--model", "todo",
    "--out_root", "inference_test",
]

# Optional
# cmd += ["--max_instances", "5"]
# cmd += ["--tag", "attempt1"]
# cmd += ["--save_prompt"]
# cmd += ["--temperature", "0.2"]
# cmd += ["--top_p", "0.95"]
# cmd += ["--max_tokens", "4000"]

print("Running:", " ".join(cmd))
print("Using python:", sys.executable)

result = subprocess.run(cmd, capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

## 3 · Path B — Agentic inference with OpenCode

This path does not use the oracle/BM25 prompt-building steps in Path A. Before running, start Docker and confirm the dataset rows include `baseline_sha` and `docker.baseline` metadata. Configure the provider/model in `opencode.docker.json` and set its required API key in your shell or project `.env`; do not put credentials in this notebook. The example config uses an OpenAI-compatible provider. OpenCode runs in Docker, so no host provider SDK extra is needed.

### Build task-only prompts

Run from the repository root:

```bash
uv run --locked python -m swe_opt.prep.prompt.build_task_only_prompts \
  --dataset_path SWE-Opt/SWE-Opt \
  --out_root data/prompt \
  --run_id quickstart_agentic
```

The builder prints the prompt run directory under `data/prompt/`. Use that exact directory (it contains `PROMPT_META.json` and `instances/`, and its name includes a short hash) in the next command.

### Run one scenario as a smoke test

Replace `<actual-prompt-run-directory>` with the directory printed above:

```bash
uv run --locked python -m swe_opt.inference.generate_agentic_patches \
  --prompts_run_dir "data/prompt/<actual-prompt-run-directory>" \
  --docker_opencode_config opencode.docker.json \
  --out_root inference/agent \
  --run_id quickstart_agentic \
  --max_instances 1
```

The inference runner prints its output directory under `inference/agent/` (the directory name includes a short hash). Inspect `completion.txt` and `agent_transcript.txt` inside its `instances/` folder. Generation completion does not establish correctness or performance; use that exact printed directory as section 4’s `--inference_run_dir` to evaluate the patch. Remove `--max_instances 1` for a full run. Use new run IDs when changing settings; the prompt builder refuses to overwrite an existing prompt run, and completed inference scenarios are skipped on reruns.

## 4 · Evaluate generated patches

Runs the evaluation pipeline on the generated inference outputs.

For each scenario, the harness reconstructs the required setup, builds the corresponding Docker images, executes both the **baseline** and **LLM-generated** containers, and compares the results.

Depending on the configuration, the harness can also run the **gold/reference solution**.

The evaluation pipeline checks execution behavior, compares outcomes across versions, and collects scenario-level results in a reproducible containerized environment.

All outputs such as logs, reports, and evaluation artifacts are stored under: results/evals/<run_id>/

> **Note on running the evaluation harness**
>
> The notebook prints the full command used to run the evaluation harness.  
> For better performance and stability, we recommend copying this printed command and running it directly in a **terminal** instead of executing it inside the notebook.
>
> The evaluation pipeline builds Docker images, launches containers, and runs multiple scenario executions. Running these steps from a terminal is typically more stable and easier to monitor than executing them inside Jupyter.

> **Note on performance measurements**
>
> Even though the evaluation is executed inside Docker containers to improve reproducibility, performance measurements (e.g., runtime or resource usage) may still vary across machines. Differences in hardware, CPU architecture, background system load, operating system behavior, and container runtime environments can lead to small variations in measured results.
>
> Therefore, performance comparisons should primarily be interpreted **relative to the baseline within the same run and environment**, rather than as absolute measurements across different machines.

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "-m",
    "swe_opt.harness.run_evaluation",
    "--inference_run_dir", "inference/<run_id>",   # replace with your actual inference output directory, e.g. inference_test
    "--results_root", "results",
]

# Optional
# cmd += ["--scenario_id", "pandas-50778"]   # run a single scenario
# cmd += ["--with-gold-solution"]            # also run reference/oracle solution
# cmd += ["--force_rebuild"]                 # force Docker image rebuild
# cmd += ["--skip-baseline-if-done"]         # skip baseline run if results already exist
# cmd += ["--skip-reference-if-done"]        # skip reference run if results already exist

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

## 5 · Aggregation and reporting

This script aggregates one completed experiment run and produces:

- gate-funnel statistics
- per-metric improvement/regression/conflict summaries
- overall aggregate statistics
- `experiment_summary.json`

Performance numbers should be interpreted relative to the baseline within the same experiment environment, not as absolute machine-independent measurements.

In [ ]:
import subprocess
import sys
from pathlib import Path


EXPERIMENT_DIR = Path("outputs/experiments/latest")
OUTPUT_PATH = EXPERIMENT_DIR / "experiment_summary.json"

cmd = [
    sys.executable,
    "-m",
    "swe_opt.reporting.aggregate_experiments",
    "--experiment_dir", str(EXPERIMENT_DIR),
    "--output_path", str(OUTPUT_PATH),
    "--print_report",
]

print("Running:", " ".join(cmd))

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(
        f"Experiment aggregation failed with exit code {result.returncode}"
    )

print(f"\nSaved summary to: {OUTPUT_PATH}")